# Feature Engineering and Feature Selection

This notebook contains the project section requested for the final ML presentation.


# Heavy Vehicle Failure Prediction Through Intelligent Sensor Data

## Complete Machine Learning Project Notebook

**Task:** Binary classification of APS-related failure in Scania heavy vehicles  
**Target:** `class` (`neg` = no APS-related failure, `pos` = APS-related failure)

This notebook contains the implementation, preprocessing, EDA, model training, tuning, evaluation, comparison, and final prediction demonstration required for the project review. Execute all cells from top to bottom before submission so the outputs and figures are visible.

The data contains 76,000 records, 170 sensor features, and a highly imbalanced target. Because missing a real failure can be costly, recall and F1-score are considered together with accuracy and ROC-AUC.


## 1. Import Libraries

The pipeline objects used later ensure that imputation and scaling are learned only from the training portion of the data. This prevents test-data leakage.


In [ ]:
# If using Google Colab and a package is missing, run this once:
# !pip install -q scikit-learn matplotlib seaborn joblib

import os
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
    cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


## 2. Load the Dataset

The original combined CSV is used because it contains both sensor values and the target label. The value `na` is converted to a real missing value while loading.


In [ ]:
candidate_paths = [
    Path("dataset/APS_Scania_Complete_Dataset.csv"),
    Path("APS_Scania_Complete_Dataset.csv"),
    Path("/content/APS_Scania_Complete_Dataset.csv"),
    Path("/content/HEAVY VEHICLE FAILURE PREDICTION THROUGH INTELLIGENT SENSOR/dataset/APS_Scania_Complete_Dataset.csv")
]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place APS_Scania_Complete_Dataset.csv in the notebook folder or update DATA_PATH.")

df = pd.read_csv(DATA_PATH, na_values=["na"])
print("Dataset path:", DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())


## 4. Data Preprocessing

Duplicate rows are checked and removed before splitting. The target is encoded as 0 and 1. Missing values are not filled manually here; the model pipelines below learn training-only median values during fitting.


In [ ]:
print("Duplicate rows before removal:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
}).sort_values("Missing_Percentage", ascending=False)

print("Duplicate rows after removal:", df.duplicated().sum())
print("Columns with missing values:", int((df.isna().sum() > 0).sum()))
print("Total missing cells:", int(df.isna().sum().sum()))
display(missing_summary.head(20))


In [ ]:
df["class"] = df["class"].map({"neg": 0, "pos": 1})

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Initial dataset shape:", (76000, 171))
print("Final dataset shape after duplicate removal:", df.shape)
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training class distribution:")
display(y_train.value_counts().rename_axis("class").to_frame("count"))
print("Testing class distribution:")
display(y_test.value_counts().rename_axis("class").to_frame("count"))


### Leakage Prevention

The test set is held out before model fitting. Every model below uses a pipeline. The median imputer and scaler are fitted on training data only and then applied to validation or test data. This prevents information from the test set entering the training process.


## 6. Feature Engineering and Feature Selection

No artificial features are created because the sensor variables are anonymized and their physical meaning is unavailable. All 170 sensor features are retained after cleaning. This avoids deleting a low-variance feature that may still be useful for failure detection.


In [ ]:
feature_names = X.columns.tolist()
print("Number of final modelling features:", len(feature_names))
print("Final feature list:")
print(feature_names)
